# **The program does:**
- Import the json file
- Read the json and process the video to extract the non-dup frames
- Write another json with this structure:
{"frameid": "frame_id",
  "gloss": "gloss_word",
"videoid": "video_id"}

In [ ]:
pip install video-kf

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Data Preparation
❌ **DO NOT RUN**
**This code will run for 11 hours on a L4 GPU.**

0. Read the json file - WLASL_v0.3.json
1. Extract all the frames of the videos in the video folder
2. Stores them in the frames folder with video id as the subfolder
3. Prepare the foundgloss.json with the gloss and non-dup frame id for training the LLaVa model
4. Prepare another file notfound.json with the list of video ids that were not found in the video folder.




In [ ]:
import json
import videokf as vf
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim


file_path = '/content/drive/MyDrive/WLASL dataset_sample/WLASL_v0.3.json'
video_path = '/content/drive/MyDrive/WLASL dataset_sample/videos'

frames_path = '/content/drive/MyDrive/WLASL dataset_sample/frames'
json_path = '/content/drive/MyDrive/WLASL dataset_sample/datafile.json'
notfound_path = '/content/drive/MyDrive/WLASL dataset_sample/notFound.json'
foundgloss_path = '/content/drive/MyDrive/WLASL dataset_sample/foundGloss.json'

datafile = []
not_found_videos = []
found_gloss = []

with open(file_path) as ipf:
    content = json.load(ipf)

# Function to calculate SSIM between two images
def compare_images(imageA, imageB):
    # Convert images to grayscale
    grayA = cv2.cvtColor(imageA, cv2.COLOR_BGR2GRAY)
    grayB = cv2.cvtColor(imageB, cv2.COLOR_BGR2GRAY)

    # Compute SSIM between two images
    score, _ = ssim(grayA, grayB, full=True)
    return score

# Load images from a directory
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename))
        if img is not None:
            images.append((filename, img))
    return images

# Function to find non-duplicate images using serial comparison
def find_non_duplicate_images_serial(images):
    non_duplicate_images = [images[0][0]]  # Start with the first image as non-duplicate
    for i in range(len(images) - 1):
        score = compare_images(images[i][1], images[i + 1][1])
        if score < 0.95:  # Threshold for considering images as duplicates
            non_duplicate_images.append(images[i + 1][0])
    return non_duplicate_images

for ent in content:
    gloss_word = ent['gloss']
    print('gloss: {}'.format(gloss_word))
    for inst in ent['instances']:
        video_id = inst['video_id']
        #find the video in the drive
        path = os.path.join(video_path, f'{video_id}.mp4') # Use os.path.join to create paths
        #extract the frames in a folder named frames_path_this_loop
        #or use method = "flow", flow is to return the most still frame with respect of the previous frame of every shot sequence. Shot sequences are group of frames that start with an iframe.
        frames_path_this_loop = frames_path + "/" + video_id
        #print('key frames in: {}'.format(frames_path_this_loop))
        try:
          vf.extract_keyframes(path, method="iframes", output_dir_keyframes = frames_path_this_loop)
          images = load_images_from_folder(frames_path_this_loop)
          non_duplicate_images = find_non_duplicate_images_serial(images)
          found_gloss.append(gloss_word)
          for img in non_duplicate_images:
              keyframe_path = os.path.join(frames_path_this_loop, img)
              data = {"gloss": gloss_word, "frameid": keyframe_path, "videoid": video_id}
              datafile.append(data)
              #print(keyframe_path)
        except Exception as e:
          not_found_videos.append(video_id)
          #print(f"Error processing video {video_id}: {e}")
          continue
        #write the json file and the notfound videos
        with open(json_path, 'w') as f:
             json.dump(datafile, f)
        with open(notfound_path, 'w') as f:
             json.dump(not_found_videos, f)
        with open(foundgloss_path, 'w') as f:
             json.dump(found_gloss, f)





Streaming output truncated to the last 5000 lines.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
gloss: pneumonia
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
gloss: politics
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
gloss: position
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
gloss: pound
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
gloss: pour
Iframes successfully extracted.
Iframes successfully extracted.
Iframes successfully extracted.
Iframes suc

In [ ]:
# prompt: load an array from a file and remove duplicates

import numpy as np
import json

foundgloss_path = '/content/drive/MyDrive/WLASL dataset/foundGloss.json'
unique_foundgloss_path = '/content/drive/MyDrive/WLASL dataset/uniquefoundGloss.json'

def load_and_remove_duplicates(foundgloss_path):
  """Loads an array from a file and removes duplicate elements.

  Args:
    file_path: Path to the file containing the array.

  Returns:
    A NumPy array with duplicate elements removed.
  """
  try:
    with open(foundgloss_path, 'r') as f:
      data = json.load(f)
      unique_array = list(dict.fromkeys(data))  # Convert to dict keys to remove duplicates
      return unique_array
  except FileNotFoundError:
    print(f"File not found: {foundgloss_path}")
    return None

# Example usage:

unique_array = load_and_remove_duplicates(foundgloss_path)
with open(unique_foundgloss_path, 'w') as f:
             json.dump(unique_array, f)

if unique_array is not None:
  print("Array with duplicates removed:", unique_array)
  print("Number of items in the array:", len(unique_array))


Array with duplicates removed: ['book', 'drink', 'computer', 'before', 'chair', 'go', 'clothes', 'who', 'candy', 'cousin', 'deaf', 'fine', 'help', 'no', 'thin', 'walk', 'year', 'yes', 'all', 'black', 'cool', 'finish', 'hot', 'like', 'many', 'mother', 'now', 'orange', 'table', 'thanksgiving', 'what', 'woman', 'bed', 'blue', 'bowling', 'can', 'dog', 'family', 'fish', 'graduate', 'hat', 'hearing', 'kiss', 'language', 'later', 'man', 'shirt', 'study', 'tall', 'white', 'wrong', 'accident', 'apple', 'bird', 'change', 'color', 'corn', 'cow', 'dance', 'dark', 'doctor', 'eat', 'enjoy', 'forget', 'give', 'last', 'meet', 'pink', 'pizza', 'play', 'school', 'secretary', 'short', 'time', 'want', 'work', 'africa', 'basketball', 'birthday', 'brown', 'but', 'cheat', 'city', 'cook', 'decide', 'full', 'how', 'jacket', 'letter', 'medicine', 'need', 'paint', 'paper', 'pull', 'purple', 'right', 'same', 'son', 'tell', 'thursday', 'visit', 'wait', 'water', 'wife', 'yellow', 'backpack', 'bar', 'brother', 'cat'